# 06 — Migration Reconciliation (Source vs Target)

**The migration-QA centerpiece.** Validates the migration of the CR
businesses registry from the "on-prem source" into Databricks by
comparing `raw.cr_businesses_source` vs `raw.cr_businesses_target`.

Reconciliation techniques (each maps to a JD responsibility):
1. **Row count reconciliation** — totals, delta, % variance
2. **Schema drift detection** — columns added/dropped/retyped
3. **Key-set reconciliation** — rows in source-not-target (lost) /
   target-not-source (injected) / duplicated
4. **Hash/checksum reconciliation** — row-level MD5 over shared columns
5. **Aggregate reconciliation** — sum/min/max/distinct on key columns
   (catches silent value drift that counts & key-sets miss)
6. **datacompy field-level diff** — column-by-column on a sample

Outputs:
- `dq.migration_defects` — defect log with root-cause hypothesis + severity
- A reconciliation report (printed; mirrors the deliverable format)
- Scored against `raw.cr_migration_manifest` (did QA catch what we broke?)

## Install datacompy into the notebook session
Serverless compute has its own Python env — it does NOT see the local
`.venv`. Libraries the notebook imports must be installed where the
code executes. `%pip install` is session-scoped; restart the Python
process after install so the new package is importable. This MUST be
the first executable cell.

In [0]:
%pip install datacompy==0.16.*

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# ── Resolve runtime config ─────────────────────────────────────
def _load_pipeline_config():
    candidates = [
        "fieldops_dq.audit.pipeline_config",
        "workspace.fieldops_audit.pipeline_config",
    ]
    for tbl in candidates:
        try:
            rows = spark.table(tbl).collect()
            cfg = {row["config_key"]: row["config_value"] for row in rows}
            print(f"Loaded config from {tbl}")
            return cfg
        except Exception:
            continue
    raise RuntimeError("pipeline_config not found. Run notebook 01 first.")

_cfg    = _load_pipeline_config()
CATALOG = _cfg["CATALOG"]
RAW     = _cfg["RAW"]
DQ      = _cfg.get("DQ", "dq")
print(f"  CATALOG={CATALOG} RAW={RAW} DQ={DQ}")

def r(t):  return f"{CATALOG}.{RAW}.{t}"
def dq(t): return f"{CATALOG}.{DQ}.{t}"

Loaded config from workspace.fieldops_audit.pipeline_config
  CATALOG=workspace RAW=fieldops_raw DQ=fieldops_dq


In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

RUN_ID  = f"MIG-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}-{str(uuid.uuid4())[:6].upper()}"
NOW     = datetime.now(timezone.utc)
KEY     = "id"                       # business key shared by source & target

# ── Guard: notebook 05 must have run first ────────────────────
# 06 reconciles tables that 05 generates. Running 06 alone yields a
# cryptic TABLE_OR_VIEW_NOT_FOUND. Fail fast with an actionable message.
for _t in ("cr_businesses_source", "cr_businesses_target", "cr_migration_manifest"):
    try:
        spark.table(r(_t)).limit(1).count()
    except Exception:
        raise RuntimeError(
            f"Required table {r(_t)} not found. "
            "Run notebook 05_generate_migration_data FIRST (Run all, let it "
            "finish), then re-run this notebook. 06 reconciles what 05 builds."
        )

src = spark.table(r("cr_businesses_source"))
tgt = spark.table(r("cr_businesses_target"))

# Accumulate defects here; written to dq.migration_defects at the end.
defects = []
def log_defect(table, column, dtype, sample_src, sample_tgt, severity, root_cause):
    defects.append({
        "defect_id":   f"{RUN_ID}-{len(defects)+1:04d}",
        "run_id":      RUN_ID,
        "table_name":  table,
        "column_name": column,
        "defect_type": dtype,
        "source_value": None if sample_src is None else str(sample_src)[:200],
        "target_value": None if sample_tgt is None else str(sample_tgt)[:200],
        "severity":    severity,
        "root_cause_hypothesis": root_cause,
        "detected_at": NOW,
    })

print(f"Run: {RUN_ID}")
print(f"Source: {r('cr_businesses_source')}")
print(f"Target: {r('cr_businesses_target')}")

Run: MIG-20260518-031539-70BA48
Source: workspace.fieldops_raw.cr_businesses_source
Target: workspace.fieldops_raw.cr_businesses_target


## Check 1 — Row count reconciliation

In [0]:
src_n = src.count()
tgt_n = tgt.count()
delta = tgt_n - src_n
pct   = (delta / src_n * 100) if src_n else 0.0

print("── ROW COUNT RECONCILIATION ──────────────────────────")
print(f"  source rows : {src_n:,}")
print(f"  target rows : {tgt_n:,}")
print(f"  delta       : {delta:+,}  ({pct:+.3f}%)")

if delta != 0:
    log_defect("cr_businesses", "(table)", "ROW_COUNT_MISMATCH",
               src_n, tgt_n,
               "CRITICAL" if abs(pct) > 1 else "HIGH",
               "Net row delta. Combination of dropped rows (data loss) and "
               "duplicated rows (non-idempotent load). See key-set check.")
    print("  [✗] FAIL — counts differ")
else:
    print("  [✓] PASS")

── ROW COUNT RECONCILIATION ──────────────────────────
  source rows : 100,000
  target rows : 99,598
  delta       : -402  (-0.402%)
  [✗] FAIL — counts differ


## Check 2 — Schema drift detection

In [0]:
src_schema = {f.name: f.dataType.simpleString() for f in src.schema.fields}
tgt_schema = {f.name: f.dataType.simpleString() for f in tgt.schema.fields}

dropped_cols = sorted(set(src_schema) - set(tgt_schema))
added_cols   = sorted(set(tgt_schema) - set(src_schema))
common       = sorted(set(src_schema) & set(tgt_schema))
retyped      = [c for c in common if src_schema[c] != tgt_schema[c]]

print("── SCHEMA DRIFT DETECTION ────────────────────────────")
print(f"  source columns : {len(src_schema)}")
print(f"  target columns : {len(tgt_schema)}")
print(f"  dropped in target : {dropped_cols or '(none)'}")
print(f"  added in target   : {added_cols or '(none)'}")
print(f"  retyped           : {retyped or '(none)'}")

for c in dropped_cols:
    log_defect("cr_businesses", c, "SCHEMA_DRIFT_COLUMN_DROPPED",
               src_schema[c], None, "CRITICAL",
               "Column present in source, absent in target — migration "
               "mapping dropped it. Data unrecoverable from target alone.")
for c in added_cols:
    log_defect("cr_businesses", c, "SCHEMA_DRIFT_COLUMN_ADDED",
               None, tgt_schema[c], "LOW",
               "New column in target not in source — likely migration "
               "metadata (lineage). Acceptable if intentional & documented.")
for c in retyped:
    log_defect("cr_businesses", c, "SCHEMA_DRIFT_TYPE_CHANGED",
               src_schema[c], tgt_schema[c], "HIGH",
               "Column datatype changed during migration — risk of silent "
               "coercion / precision loss.")

print("  [✗] FAIL" if (dropped_cols or added_cols or retyped) else "  [✓] PASS")

── SCHEMA DRIFT DETECTION ────────────────────────────
  source columns : 10
  target columns : 11
  dropped in target : ['linkedin_url']
  added in target   : ['migrated_at', 'source_system']
  retyped           : (none)
  [✗] FAIL


## Check 3 — Key-set reconciliation
Which business keys were lost, injected, or duplicated.

In [0]:
src_keys = src.select(KEY).distinct()
tgt_keys = tgt.select(KEY).distinct()

lost     = src_keys.subtract(tgt_keys)          # in source, not target
injected = tgt_keys.subtract(src_keys)          # in target, not source
lost_n     = lost.count()
injected_n = injected.count()

# duplicates in target: keys appearing >1 time
tgt_dupes = (tgt.groupBy(KEY).count()
                .filter(F.col("count") > 1))
dupe_keys_n = tgt_dupes.count()
dupe_rows_n = (tgt_dupes.agg(F.sum(F.col("count") - F.lit(1))
                  .alias("x")).collect()[0]["x"]) or 0

print("── KEY-SET RECONCILIATION ────────────────────────────")
print(f"  keys lost (source→target missing) : {lost_n:,}")
print(f"  keys injected (target only)       : {injected_n:,}")
print(f"  duplicated keys in target         : {dupe_keys_n:,} "
      f"({dupe_rows_n:,} surplus rows)")

if lost_n:
    sample = [r0[KEY] for r0 in lost.limit(5).collect()]
    log_defect("cr_businesses", KEY, "ROWS_LOST_IN_MIGRATION",
               f"{lost_n} keys e.g. {sample}", None, "CRITICAL",
               "Business keys present in source but missing in target — "
               "rows lost during migration extract/load.")
if injected_n:
    sample = [r0[KEY] for r0 in injected.limit(5).collect()]
    log_defect("cr_businesses", KEY, "ROWS_INJECTED_IN_TARGET",
               None, f"{injected_n} keys e.g. {sample}", "HIGH",
               "Keys in target with no source origin — phantom/orphan rows.")
if dupe_keys_n:
    sample = [r0[KEY] for r0 in tgt_dupes.limit(5).collect()]
    log_defect("cr_businesses", KEY, "DUPLICATE_KEYS_IN_TARGET",
               None, f"{dupe_keys_n} dup keys e.g. {sample}", "HIGH",
               "Duplicate business keys in target — non-idempotent load "
               "(partial re-run appended rows).")

print("  [✗] FAIL" if (lost_n or injected_n or dupe_keys_n) else "  [✓] PASS")

── KEY-SET RECONCILIATION ────────────────────────────
  keys lost (source→target missing) : 502
  keys injected (target only)       : 0
  duplicated keys in target         : 100 (100 surplus rows)
  [✗] FAIL


## Check 4 — Row-level hash reconciliation
MD5 over the **shared** columns. For keys present in both, a hash
mismatch means at least one shared field changed value.

In [0]:
# Hash only columns common to both (exclude schema-drift cols & dup noise).
hash_cols = [c for c in common if c != KEY]

def with_rowhash(df, cols, alias):
    expr = F.md5(F.concat_ws("||", *[
        F.coalesce(F.col(c).cast("string"), F.lit("∅")) for c in cols
    ]))
    return df.select(F.col(KEY).alias(f"{alias}_key"), expr.alias(f"{alias}_hash"))

# de-dup target to one row per key for a fair 1:1 hash comparison
tgt_dedup = tgt.dropDuplicates([KEY])

sh = with_rowhash(src, hash_cols, "s")
th = with_rowhash(tgt_dedup, hash_cols, "t")

joined = sh.join(th, sh.s_key == th.t_key, "inner")
mismatch = joined.filter(F.col("s_hash") != F.col("t_hash"))
mismatch_n = mismatch.count()
compared_n = joined.count()

print("── ROW-LEVEL HASH RECONCILIATION ─────────────────────")
print(f"  keys compared (in both)   : {compared_n:,}")
print(f"  hash mismatches (changed) : {mismatch_n:,}")

if mismatch_n:
    sample = [row["s_key"] for row in mismatch.limit(5).collect()]
    log_defect("cr_businesses", f"shared({len(hash_cols)} cols)",
               "ROW_VALUE_MISMATCH", None,
               f"{mismatch_n} rows e.g. {sample}", "HIGH",
               "Row hash differs for keys in both — value(s) changed during "
               "migration (taxonomy rename / casing / truncation / numeric).")
    print("  [✗] FAIL")
else:
    print("  [✓] PASS")

── ROW-LEVEL HASH RECONCILIATION ─────────────────────
  keys compared (in both)   : 99,498
  hash mismatches (changed) : 17,124
  [✗] FAIL


## Check 5 — Aggregate reconciliation
Column-level aggregates catch *silent* drift the row count can't see
(e.g. a few `founded` years shifted, an industry value renamed).

In [0]:
print("── AGGREGATE RECONCILIATION ──────────────────────────")

# 5a. numeric: founded — sum / min / max
s_agg = src.agg(F.sum("founded").alias("s_sum"),
                F.min("founded").alias("s_min"),
                F.max("founded").alias("s_max")).collect()[0]
# compare against de-duplicated target & only keys that exist in source,
# so duplicates/injected rows don't pollute the aggregate comparison
tgt_matched = tgt_dedup.join(src.select(KEY), KEY, "inner")
t_agg = tgt_matched.agg(F.sum("founded").alias("t_sum"),
                        F.min("founded").alias("t_min"),
                        F.max("founded").alias("t_max")).collect()[0]

print(f"  founded SUM  src={s_agg['s_sum']:,}  tgt={t_agg['t_sum']:,}  "
      f"Δ={t_agg['t_sum']-s_agg['s_sum']:+,}")
print(f"  founded MIN  src={s_agg['s_min']}  tgt={t_agg['t_min']}")
print(f"  founded MAX  src={s_agg['s_max']}  tgt={t_agg['t_max']}")

if s_agg["s_sum"] != t_agg["t_sum"]:
    log_defect("cr_businesses", "founded", "AGGREGATE_SUM_MISMATCH",
               s_agg["s_sum"], t_agg["t_sum"], "HIGH",
               "SUM(founded) differs across matched keys — silent numeric "
               "drift; individual values altered during migration.")

# 5b. categorical: industry distinct-value set
s_inds = {row["industry"] for row in src.select("industry").distinct().collect()}
t_inds = {row["industry"] for row in
          tgt_dedup.select("industry").distinct().collect()}
only_src = sorted(s_inds - t_inds)
only_tgt = sorted(t_inds - s_inds)
print(f"  industry values only in SOURCE : {only_src or '(none)'}")
print(f"  industry values only in TARGET : {only_tgt or '(none)'}")

if only_src or only_tgt:
    log_defect("cr_businesses", "industry", "CATEGORICAL_DOMAIN_DRIFT",
               str(only_src), str(only_tgt), "HIGH",
               "Distinct industry domain changed — taxonomy not preserved "
               "(values renamed in migration mapping).")
    print("  [✗] FAIL")
else:
    print("  [✓] PASS")

── AGGREGATE RECONCILIATION ──────────────────────────
  founded SUM  src=198,696,652  tgt=197,700,117  Δ=-996,535
  founded MIN  src=1950  tgt=1950
  founded MAX  src=2024  tgt=2025
  industry values only in SOURCE : ['Information Technology', 'Tourism & Hospitality']
  industry values only in TARGET : ['IT', 'Tourism']
  [✗] FAIL


## Check 6 — datacompy field-level diff (sample)
Industry-standard source-vs-target comparison library. We run it on a
sample of matched keys for a column-by-column mismatch breakdown.

In [0]:
import datacompy
import pandas as pd

SAMPLE_N = 5000   # sample keeps the pandas compare fast & memory-safe

shared_cols = [KEY] + hash_cols
src_s = (src.select(*shared_cols)
            .orderBy(KEY).limit(SAMPLE_N).toPandas())
tgt_s = (tgt_dedup.select(*shared_cols)
            .join(src.select(KEY), KEY, "inner")
            .orderBy(KEY).limit(SAMPLE_N).toPandas())

compare = datacompy.Compare(
    src_s, tgt_s,
    join_columns=KEY,
    df1_name="source",
    df2_name="target",
)

print("── DATACOMPY FIELD-LEVEL DIFF (sample) ───────────────")
print(f"  sample size           : {len(src_s):,} matched keys")
print(f"  rows match            : {compare.count_matching_rows():,}")
print(f"  all columns match?    : {compare.matches()}")

# compare.column_stats is the documented per-column summary: a list of
# dicts each with 'column', 'unequal_cnt', 'dtype1', 'dtype2', etc.
# sample_mismatch(col) returns a small DataFrame of differing rows for
# that column with '{col}_df1' / '{col}_df2' value columns (datacompy
# lowercases names by default, suffixes are df1/df2 — NOT the names).
for cs in compare.column_stats:
    col = cs["column"]
    n_unequal = int(cs.get("unequal_cnt", 0))
    if n_unequal and col != KEY:
        s_ex = t_ex = None
        try:
            sm = compare.sample_mismatch(col, sample_count=1)
            if len(sm):
                s_ex = sm.iloc[0].get(f"{col}_df1")
                t_ex = sm.iloc[0].get(f"{col}_df2")
        except Exception:
            pass
        print(f"  column '{col}': {n_unequal:,} mismatched "
              f"(e.g. {s_ex!r} → {t_ex!r})")
        log_defect("cr_businesses", col, "FIELD_VALUE_MISMATCH",
                   s_ex, t_ex, "MEDIUM",
                   f"datacompy: {n_unequal} sampled rows differ in '{col}'. "
                   "Field-level corruption introduced during migration.")

# Human-readable datacompy report — the SAS PROC COMPARE-style summary.
# This is exactly the artifact a QA analyst attaches to a recon report.
print("\n── datacompy report (first 40 lines) ─────────────────")
for line in compare.report().splitlines()[:40]:
    print("  " + line)

Please note that you are missing the optional dependency: fugue. If you need to use this functionality it must be installed.
Please note that you are missing the optional dependency: snowflake. If you need to use this functionality it must be installed.


── DATACOMPY FIELD-LEVEL DIFF (sample) ───────────────
  sample size           : 5,000 matched keys
  rows match            : 4,104
  all columns match?    : False
  column 'founded': 10 mismatched (e.g. None → None)
  column 'industry': 850 mismatched (e.g. None → None)
  column 'name': 16 mismatched (e.g. None → None)
  column 'website': 16 mismatched (e.g. None → None)

── datacompy report (first 40 lines) ─────────────────
  DataComPy Comparison
  --------------------
  
  DataFrame Summary
  -----------------
  
    DataFrame  Columns  Rows
  0    source        9  5000
  1    target        9  5000
  
  Column Summary
  --------------
  
  Number of columns in common: 9
  Number of columns in source but not in target: 0 []
  Number of columns in target but not in source: 0 []
  
  Row Summary
  -----------
  
  Matched on: id
  Any duplicates on match values: No
  Absolute Tolerance: 0
  Relative Tolerance: 0
  Number of rows in common: 4,975
  Number of rows in source but not in t

## Persist defect log + reconciliation summary

In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                               TimestampType)

defect_schema = StructType([
    StructField("defect_id",              StringType(),    True),
    StructField("run_id",                 StringType(),    True),
    StructField("table_name",             StringType(),    True),
    StructField("column_name",            StringType(),    True),
    StructField("defect_type",            StringType(),    True),
    StructField("source_value",           StringType(),    True),
    StructField("target_value",           StringType(),    True),
    StructField("severity",               StringType(),    True),
    StructField("root_cause_hypothesis",  StringType(),    True),
    StructField("detected_at",            TimestampType(), True),
])

if defects:
    ddf = spark.createDataFrame(defects, schema=defect_schema)
    ddf.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(dq("migration_defects"))
    print(f"Defect log written: {dq('migration_defects')} "
          f"= {len(defects)} defects")
else:
    print("No defects — migration reconciled clean.")

Defect log written: workspace.fieldops_dq.migration_defects = 13 defects


In [0]:
# Score against the injected-defect manifest (the test oracle):
# did reconciliation catch each class we deliberately broke?
try:
    man = spark.table(r("cr_migration_manifest")).collect()
    found_types = {d["defect_type"] for d in defects}
    mapping = {
        "row_loss":      "ROWS_LOST_IN_MIGRATION",
        "taxonomy":      "CATEGORICAL_DOMAIN_DRIFT",
        "corruption":    "ROW_VALUE_MISMATCH",
        "numeric_drift": "AGGREGATE_SUM_MISMATCH",
        "schema_drift":  "SCHEMA_DRIFT_COLUMN_DROPPED",
        "duplicates":    "DUPLICATE_KEYS_IN_TARGET",
    }
    print("\n── RECONCILIATION SCORECARD (vs injected manifest) ───")
    caught = 0
    for row in man:
        cls = row["defect_class"]
        want = mapping.get(cls)
        hit  = want in found_types
        caught += hit
        print(f"  {cls:<14} {'✓ CAUGHT' if hit else '✗ MISSED':<10} "
              f"— {row['description']}")
    print(f"\n  Detection rate: {caught}/{len(man)} injected defect classes")
except Exception as e:
    print(f"(manifest scoring skipped: {type(e).__name__})")


── RECONCILIATION SCORECARD (vs injected manifest) ───
  row_loss       ✓ CAUGHT   — rows dropped (id % 199 == 0)
  taxonomy       ✓ CAUGHT   — industry values renamed
  corruption     ✓ CAUGHT   — name/website corrupted (id % 311 == 0)
  numeric_drift  ✓ CAUGHT   — founded += 1 (id % 467 == 0)
  schema_drift   ✓ CAUGHT   — linkedin_url dropped; 2 cols added
  duplicates     ✓ CAUGHT   — rows duplicated (id % 991 == 0)

  Detection rate: 6/6 injected defect classes


In [0]:
print("=" * 58)
print(f"  MIGRATION RECONCILIATION COMPLETE — {RUN_ID}")
print("=" * 58)
print(f"  source rows         : {src_n:,}")
print(f"  target rows         : {tgt_n:,}  ({tgt_n-src_n:+,})")
print(f"  total defects logged: {len(defects)}")
by_sev = {}
for d in defects:
    by_sev[d["severity"]] = by_sev.get(d["severity"], 0) + 1
for sev in ("CRITICAL", "HIGH", "MEDIUM", "LOW"):
    if sev in by_sev:
        print(f"    {sev:<9}: {by_sev[sev]}")
print(f"\n  Defect log: {dq('migration_defects')}")
print("  Next: 07 generates the reconciliation report document")

  MIGRATION RECONCILIATION COMPLETE — MIG-20260518-031539-70BA48
  source rows         : 100,000
  target rows         : 99,598  (-402)
  total defects logged: 13
    CRITICAL : 2
    HIGH     : 5
    MEDIUM   : 4
    LOW      : 2

  Defect log: workspace.fieldops_dq.migration_defects
  Next: 07 generates the reconciliation report document
